# Full Snapshot Smoke Tests

Validates the snapshot after export by checking:
1. ID format correctness for all entities
2. Entity counts vs OpenAlex API
3. Works field completeness vs API response schema
4. Sample works field-by-field comparison against API
5. Authorships with affiliations at expected rates
6. Works nested structure spot checks
7. Null checks on required fields
8. Duplicate ID check
9. S3 manifest validation
10. S3 partition path format
11. Abstract truncation JSON validity
12. Updated date recency
13. Keywords & SDGs population rates

In [ ]:
import json
import re
import requests
import time
from pyspark.sql import functions as F

API_BASE = "https://api.openalex.org"
POLITE_EMAIL = "casey@ourresearch.org"

results = []

def record(test_name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"test": test_name, "passed": passed, "detail": detail})
    icon = "OK" if passed else "XX"
    print(f"  [{icon}] {test_name}: {detail}")

def api_get(path, params=None):
    """Call OpenAlex API with polite pool and retry."""
    url = f"{API_BASE}/{path}"
    p = {"mailto": POLITE_EMAIL}
    if params:
        p.update(params)
    for attempt in range(3):
        try:
            r = requests.get(url, params=p, timeout=30)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if attempt < 2:
                time.sleep(2 ** attempt)
            else:
                raise

works_df = spark.read.table("openalex.works.openalex_works_snapshot")
print("Setup complete.")

## Test 1: ID Format Validation
Every entity must have IDs matching the expected OpenAlex URL pattern.

In [ ]:
print("=" * 60)
print("TEST 1: ID Format Validation")
print("=" * 60)

ENTITY_ID_SPECS = [
    ("works",        "openalex.works.openalex_works_snapshot",              r"^https://openalex\.org/W\d+$"),
    ("authors",      "openalex.authors.openalex_authors_snapshot",          r"^https://openalex\.org/A\d+$"),
    ("institutions", "openalex.institutions.openalex_institutions_snapshot", r"^https://openalex\.org/I\d+$"),
    ("sources",      "openalex.sources.openalex_sources_snapshot",          r"^https://openalex\.org/S\d+$"),
    ("publishers",   "openalex.publishers.openalex_publishers_snapshot",    r"^https://openalex\.org/P\d+$"),
    ("funders",      "openalex.funders.openalex_funders_snapshot",          r"^https://openalex\.org/F\d+$"),
    ("topics",       "openalex.common.openalex_topics_snapshot",            r"^https://openalex\.org/T\d+$"),
    ("concepts",     "openalex.common.openalex_concepts_snapshot",          r"^https://openalex\.org/C\d+$"),
    ("awards",       "openalex.awards.openalex_awards_snapshot",            r"^https://openalex\.org/G\d+$"),
    ("keywords",     "openalex.common.openalex_keywords_snapshot",          r"^https://openalex\.org/keywords/.+$"),
]

# Works ID check via single aggregation (avoids extra full scans)
works_pattern = r"^https://openalex\.org/W\d+$"
works_id_stats = works_df.agg(
    F.count("*").alias("total"),
    F.count(F.when(F.col("id").isNull(), 1)).alias("null_ids"),
    F.count(F.when(~F.col("id").rlike(works_pattern), 1)).alias("bad_ids"),
).first()
works_total = works_id_stats["total"]
passed = (works_id_stats["bad_ids"] == 0) and (works_id_stats["null_ids"] == 0)
detail = f"{works_total:,} records, {works_id_stats['null_ids']} null IDs, {works_id_stats['bad_ids']} malformed IDs (pattern: {works_pattern})"
record("id_format_works", passed, detail)

for entity_name, table, pattern in ENTITY_ID_SPECS:
    if table == "openalex.works.openalex_works_snapshot":
        continue  # already handled above
    df = spark.read.table(table)
    total = df.count()
    bad_ids = df.filter(~F.col("id").rlike(pattern)).count()
    null_ids = df.filter(F.col("id").isNull()).count()
    passed = (bad_ids == 0) and (null_ids == 0)
    detail = f"{total:,} records, {null_ids} null IDs, {bad_ids} malformed IDs (pattern: {pattern})"
    record(f"id_format_{entity_name}", passed, detail)

# Subfields, fields, domains use openalex.org URLs with the hierarchy prefix
HIERARCHY_SPECS = [
    ("subfields", "openalex.common.openalex_subfields_snapshot", r"^https://openalex\.org/subfields/\d+$"),
    ("fields",    "openalex.common.openalex_fields_snapshot",    r"^https://openalex\.org/fields/\d+$"),
    ("domains",   "openalex.common.openalex_domains_snapshot",   r"^https://openalex\.org/domains/\d+$"),
]

for entity_name, table, pattern in HIERARCHY_SPECS:
    df = spark.read.table(table)
    total = df.count()
    bad_ids = df.filter(~F.col("id").rlike(pattern)).count()
    null_ids = df.filter(F.col("id").isNull()).count()
    passed = (bad_ids == 0) and (null_ids == 0)
    detail = f"{total:,} records, {null_ids} null IDs, {bad_ids} malformed IDs (pattern: {pattern})"
    record(f"id_format_{entity_name}", passed, detail)

## Test 2: Entity Counts vs API
Snapshot record counts should be within 5% of the live API counts.

In [ ]:
print("=" * 60)
print("TEST 2: Entity Counts vs API")
print("=" * 60)

COUNT_TOLERANCE = 0.05  # 5% tolerance

ENTITY_COUNT_SPECS = [
    ("works",        "openalex.works.openalex_works_snapshot",              "works"),
    ("authors",      "openalex.authors.openalex_authors_snapshot",          "authors"),
    ("institutions", "openalex.institutions.openalex_institutions_snapshot", "institutions"),
    ("sources",      "openalex.sources.openalex_sources_snapshot",          "sources"),
    ("publishers",   "openalex.publishers.openalex_publishers_snapshot",    "publishers"),
    ("funders",      "openalex.funders.openalex_funders_snapshot",          "funders"),
    ("topics",       "openalex.common.openalex_topics_snapshot",            "topics"),
    ("concepts",     "openalex.common.openalex_concepts_snapshot",          "concepts"),
    ("keywords",     "openalex.common.openalex_keywords_snapshot",          "keywords"),
]

for entity_name, table, api_endpoint in ENTITY_COUNT_SPECS:
    snapshot_count = works_total if table == "openalex.works.openalex_works_snapshot" else spark.read.table(table).count()
    # Works API excludes xpac by default; snapshot includes them
    api_params = {"per_page": "1", "select": "id"}
    if api_endpoint == "works":
        api_params["include-xpac"] = "true"
    api_resp = api_get(api_endpoint, api_params)
    api_count = api_resp["meta"]["count"]
    
    if api_count == 0:
        pct_diff = 0 if snapshot_count == 0 else float('inf')
    else:
        pct_diff = abs(snapshot_count - api_count) / api_count
    
    passed = pct_diff <= COUNT_TOLERANCE
    detail = f"snapshot={snapshot_count:,}  api={api_count:,}  diff={pct_diff:.2%}"
    record(f"count_{entity_name}", passed, detail)

## Test 3: Works Field Completeness vs API
The snapshot works table should have all the fields present in API responses.
Some fields are intentionally excluded from the snapshot (e.g. `content_urls`).

In [ ]:
print("=" * 60)
print("TEST 3: Works Field Completeness vs API")
print("=" * 60)

# Fetch a well-populated work from the API to get the canonical field list
api_work = api_get("works/W2741809807")
api_fields = set(api_work.keys())

# Get snapshot table fields from cached DataFrame
snapshot_fields = set(works_df.columns)

# Fields intentionally not in snapshot (internal API-only fields)
KNOWN_EXCLUSIONS = {"content_urls"}

# Fields in snapshot but not in API (snapshot extras)
KNOWN_EXTRAS = {"authors_count"}

missing_from_snapshot = api_fields - snapshot_fields - KNOWN_EXCLUSIONS
extra_in_snapshot = snapshot_fields - api_fields - KNOWN_EXTRAS

passed = len(missing_from_snapshot) == 0
detail = f"API has {len(api_fields)} fields, snapshot has {len(snapshot_fields)} fields"
if missing_from_snapshot:
    detail += f"\n    MISSING from snapshot: {sorted(missing_from_snapshot)}"
if extra_in_snapshot:
    detail += f"\n    EXTRA in snapshot (not in API): {sorted(extra_in_snapshot)}"
if KNOWN_EXCLUSIONS & api_fields:
    detail += f"\n    Known exclusions (OK): {sorted(KNOWN_EXCLUSIONS & api_fields)}"
record("works_field_completeness", passed, detail)

print(f"\n  API fields:      {sorted(api_fields)}")
print(f"  Snapshot fields: {sorted(snapshot_fields)}")

## Test 4: Sample Works Comparison Against API
Fetch 5 works from the API and verify key fields match the snapshot table.

In [ ]:
print("=" * 60)
print("TEST 4: Sample Works Comparison Against API")
print("=" * 60)

# Get a few random works from the cached snapshot to cross-check
random_works = (
    works_df
    .filter(F.col("doi").isNotNull())
    .filter(F.col("publication_year") >= 2020)
    .filter(F.size("authorships") > 0)
    .filter(F.col("primary_topic").isNotNull())
    .orderBy(F.rand(seed=42))
    .limit(5)
    .select("id")
    .collect()
)

sample_ids = [row["id"] for row in random_works]
print(f"  Testing {len(sample_ids)} sample works: {sample_ids}")

# Fields to compare (scalar and simple fields)
SCALAR_FIELDS = [
    "id", "doi", "title", "display_name", "publication_year",
    "language", "type", "is_retracted", "is_paratext", "is_xpac",
    "cited_by_count", "locations_count", "has_fulltext",
    "countries_distinct_count", "institutions_distinct_count",
    "referenced_works_count",
]

all_match = True
for work_url_id in sample_ids:
    # Extract the short ID (W12345) from full URL
    short_id = work_url_id.split("/")[-1]
    
    # Get from API
    try:
        api_work = api_get(f"works/{short_id}")
    except Exception as e:
        record(f"sample_work_{short_id}", False, f"API fetch failed: {e}")
        all_match = False
        continue
    
    # Get from cached snapshot
    snap_row = works_df.filter(F.col("id") == work_url_id).first()
    
    if snap_row is None:
        record(f"sample_work_{short_id}", False, "Not found in snapshot table")
        all_match = False
        continue
    
    snap_dict = snap_row.asDict(recursive=True)
    
    mismatches = []
    for field in SCALAR_FIELDS:
        api_val = api_work.get(field)
        snap_val = snap_dict.get(field)
        
        if api_val is None and snap_val is None:
            continue
        
        if str(api_val) != str(snap_val):
            mismatches.append(f"{field}: api={api_val!r} snap={snap_val!r}")
    
    # Check open_access sub-fields
    if "open_access" in api_work and "open_access" in snap_dict:
        api_oa = api_work["open_access"]
        snap_oa = snap_dict["open_access"]
        for oa_field in ["is_oa", "oa_status"]:
            if api_oa.get(oa_field) != snap_oa.get(oa_field):
                mismatches.append(f"open_access.{oa_field}: api={api_oa.get(oa_field)!r} snap={snap_oa.get(oa_field)!r}")
    
    # Check primary_topic
    if api_work.get("primary_topic") and snap_dict.get("primary_topic"):
        api_pt = api_work["primary_topic"]
        snap_pt = snap_dict["primary_topic"]
        if api_pt.get("id") != snap_pt.get("id"):
            mismatches.append(f"primary_topic.id: api={api_pt.get('id')!r} snap={snap_pt.get('id')!r}")
    
    # Check authorships count
    api_auth_count = len(api_work.get("authorships", []))
    snap_auth_count = len(snap_dict.get("authorships", []))
    if api_auth_count != snap_auth_count:
        mismatches.append(f"authorships_count: api={api_auth_count} snap={snap_auth_count}")
    
    # Check topics count
    api_topics_count = len(api_work.get("topics", []))
    snap_topics_count = len(snap_dict.get("topics", []))
    if api_topics_count != snap_topics_count:
        mismatches.append(f"topics_count: api={api_topics_count} snap={snap_topics_count}")
    
    # Check referenced_works count
    api_ref_count = len(api_work.get("referenced_works", []))
    snap_ref_count = len(snap_dict.get("referenced_works", []))
    if api_ref_count != snap_ref_count:
        mismatches.append(f"referenced_works_len: api={api_ref_count} snap={snap_ref_count}")
    
    # Check concepts count
    api_concepts_count = len(api_work.get("concepts", []))
    snap_concepts_count = len(snap_dict.get("concepts", []))
    if api_concepts_count != snap_concepts_count:
        mismatches.append(f"concepts_count: api={api_concepts_count} snap={snap_concepts_count}")
    
    # Check locations count
    api_loc_count = len(api_work.get("locations", []))
    snap_loc_count = len(snap_dict.get("locations", []))
    if api_loc_count != snap_loc_count:
        mismatches.append(f"locations_len: api={api_loc_count} snap={snap_loc_count}")
    
    passed = len(mismatches) == 0
    if not passed:
        all_match = False
    detail = f"{len(mismatches)} mismatches" + (f": {'; '.join(mismatches[:5])}" if mismatches else "")
    record(f"sample_work_{short_id}", passed, detail)
    time.sleep(0.2)  # polite rate limiting

## Test 5, 5b & 7: Authorships, Affiliations & Null Checks
All stats computed in a single aggregation pass over the works table.
Thresholds derived from actual data (stable across 1-week Delta time travel).
Non-xpac filtering gives a cleaner signal on affiliation coverage.

In [ ]:
print("=" * 60)
print("TEST 5, 5b & 7: Authorships, Affiliations & Null Checks")
print("=" * 60)

# Null check fields
REQUIRED_FIELDS = ["id", "type", "is_paratext"]
MOSTLY_REQUIRED = ["title", "publication_year", "updated_date", "created_date", "is_retracted"]

# Single aggregation pass for affiliation stats + null checks
null_aggs = [F.count(F.when(F.col(f).isNull(), 1)).alias(f"null_{f}") for f in REQUIRED_FIELDS + MOSTLY_REQUIRED]

agg_stats = works_df.agg(
    # All works
    F.count("*").alias("total"),
    F.count(F.when(F.size("authorships") > 0, 1)).alias("with_auth"),
    F.count(F.when(F.exists("authorships", lambda a: F.size(a.institutions) > 0), 1)).alias("with_affil"),
    # 2020+ works
    F.count(F.when(F.col("publication_year") >= 2020, 1)).alias("recent_total"),
    F.count(F.when((F.col("publication_year") >= 2020) & F.exists("authorships", lambda a: F.size(a.institutions) > 0), 1)).alias("recent_with_affil"),
    # Non-xpac works
    F.count(F.when(F.col("is_xpac") == False, 1)).alias("non_xpac_total"),
    F.count(F.when((F.col("is_xpac") == False) & (F.size("authorships") > 0), 1)).alias("non_xpac_with_auth"),
    F.count(F.when((F.col("is_xpac") == False) & F.exists("authorships", lambda a: F.size(a.institutions) > 0), 1)).alias("non_xpac_with_affil"),
    # Non-xpac 2020+ works
    F.count(F.when((F.col("is_xpac") == False) & (F.col("publication_year") >= 2020), 1)).alias("non_xpac_recent_total"),
    F.count(F.when((F.col("is_xpac") == False) & (F.col("publication_year") >= 2020) & (F.size("authorships") > 0), 1)).alias("non_xpac_recent_with_auth"),
    F.count(F.when((F.col("is_xpac") == False) & (F.col("publication_year") >= 2020) & F.exists("authorships", lambda a: F.size(a.institutions) > 0), 1)).alias("non_xpac_recent_with_affil"),
    F.count(F.when((F.col("is_xpac") == False) & (F.col("publication_year") >= 2020) & F.exists("authorships", lambda a: F.size(a.raw_affiliation_strings) > 0), 1)).alias("non_xpac_recent_with_ras"),
    # Null checks
    *null_aggs,
).first()

s = agg_stats.asDict()

# --- Test 5: All works ---
auth_pct = s["with_auth"] / s["total"]
affil_pct = s["with_affil"] / s["total"]
recent_affil_pct = s["recent_with_affil"] / s["recent_total"]

passed_auth = auth_pct >= 0.85
record("works_have_authorships", passed_auth,
       f"{s['with_auth']:,}/{s['total']:,} ({auth_pct:.1%}) have authorships [threshold: 85%]")

passed_affil = affil_pct >= 0.28
record("works_have_affiliations", passed_affil,
       f"{s['with_affil']:,}/{s['total']:,} ({affil_pct:.1%}) have institutional affiliations [threshold: 28%]")

passed_recent = recent_affil_pct >= 0.44
record("recent_works_affiliations", passed_recent,
       f"{s['recent_with_affil']:,}/{s['recent_total']:,} ({recent_affil_pct:.1%}) of 2020+ works have affiliations [threshold: 44%]")

# raw_affiliation_strings on affiliated works (sample-based, still separate since it needs a filter+limit)
works_sample = works_df.filter(F.exists("authorships", lambda a: F.size(a.institutions) > 0)).limit(10000)
sample_stats = works_sample.agg(
    F.count("*").alias("sample_total"),
    F.count(F.when(F.exists("authorships", lambda a: F.size(a.raw_affiliation_strings) > 0), 1)).alias("with_ras"),
).first()
ras_pct = sample_stats["with_ras"] / sample_stats["sample_total"] if sample_stats["sample_total"] > 0 else 0
passed_ras = ras_pct >= 0.95
record("raw_affiliation_strings_populated", passed_ras,
       f"{sample_stats['with_ras']:,}/{sample_stats['sample_total']:,} ({ras_pct:.1%}) of affiliated works have raw_affiliation_strings [threshold: 95%]")

# --- Test 5b: Non-xpac ---
nx_auth_pct = s["non_xpac_with_auth"] / s["non_xpac_total"]
nx_affil_pct = s["non_xpac_with_affil"] / s["non_xpac_total"]

passed_nx_auth = nx_auth_pct >= 0.88
record("non_xpac_have_authorships", passed_nx_auth,
       f"{s['non_xpac_with_auth']:,}/{s['non_xpac_total']:,} ({nx_auth_pct:.1%}) non-xpac works have authorships [threshold: 88%]")

passed_nx_affil = nx_affil_pct >= 0.36
record("non_xpac_have_affiliations", passed_nx_affil,
       f"{s['non_xpac_with_affil']:,}/{s['non_xpac_total']:,} ({nx_affil_pct:.1%}) non-xpac works have affiliations [threshold: 36%]")

nx_recent_auth_pct = s["non_xpac_recent_with_auth"] / s["non_xpac_recent_total"]
nx_recent_affil_pct = s["non_xpac_recent_with_affil"] / s["non_xpac_recent_total"]
nx_recent_ras_pct = s["non_xpac_recent_with_ras"] / s["non_xpac_recent_total"]

passed_nx_r_auth = nx_recent_auth_pct >= 0.89
record("non_xpac_2020_authorships", passed_nx_r_auth,
       f"{s['non_xpac_recent_with_auth']:,}/{s['non_xpac_recent_total']:,} ({nx_recent_auth_pct:.1%}) non-xpac 2020+ have authorships [threshold: 89%]")

passed_nx_r_affil = nx_recent_affil_pct >= 0.50
record("non_xpac_2020_affiliations", passed_nx_r_affil,
       f"{s['non_xpac_recent_with_affil']:,}/{s['non_xpac_recent_total']:,} ({nx_recent_affil_pct:.1%}) non-xpac 2020+ have affiliations [threshold: 50%]")

passed_nx_r_ras = nx_recent_ras_pct >= 0.55
record("non_xpac_2020_raw_affil_strings", passed_nx_r_ras,
       f"{s['non_xpac_recent_with_ras']:,}/{s['non_xpac_recent_total']:,} ({nx_recent_ras_pct:.1%}) non-xpac 2020+ have raw_affiliation_strings [threshold: 55%]")

# --- Test 7: Null checks (from same aggregation) ---
print()
print("=" * 60)
print("TEST 7: Null Checks on Required Fields")
print("=" * 60)

for field in REQUIRED_FIELDS:
    null_count = s[f"null_{field}"]
    passed = null_count == 0
    record(f"not_null_{field}", passed,
           f"{null_count:,}/{works_total:,} null values")

for field in MOSTLY_REQUIRED:
    null_count = s[f"null_{field}"]
    null_pct = null_count / works_total if works_total > 0 else 0
    passed = null_pct < 0.10
    record(f"mostly_not_null_{field}", passed,
           f"{null_count:,}/{works_total:,} ({null_pct:.2%}) null values")

In [ ]:
print("=" * 60)
print("TEST 6: Works Nested Structure Spot Checks")
print("=" * 60)

# Get a work with rich data for structure checking from cached df
rich_work = (
    works_df
    .filter(F.size("authorships") > 2)
    .filter(F.size("locations") > 0)
    .filter(F.col("primary_topic").isNotNull())
    .filter(F.size("concepts") > 0)
    .limit(1)
    .first()
)

if rich_work:
    rd = rich_work.asDict(recursive=True)
    
    # Check authorships structure
    expected_auth_fields = {"author", "author_position", "affiliations", "countries",
                           "raw_author_name", "is_corresponding", "raw_affiliation_strings", "institutions"}
    actual_auth_fields = set(rd["authorships"][0].keys()) if rd.get("authorships") else set()
    auth_ok = expected_auth_fields.issubset(actual_auth_fields)
    record("authorships_structure", auth_ok,
           f"expected={sorted(expected_auth_fields)}, actual={sorted(actual_auth_fields)}")
    
    # Check locations structure
    expected_loc_fields = {"id", "source", "is_oa", "is_published", "landing_page_url", "pdf_url",
                          "raw_source_name", "raw_type", "provenance", "license", "license_id",
                          "version", "is_accepted"}
    actual_loc_fields = set(rd["locations"][0].keys()) if rd.get("locations") else set()
    loc_ok = expected_loc_fields.issubset(actual_loc_fields)
    record("locations_structure", loc_ok,
           f"expected={sorted(expected_loc_fields)}, actual={sorted(actual_loc_fields)}")
    
    # Check primary_topic structure
    expected_topic_fields = {"id", "display_name", "score", "subfield", "field", "domain"}
    actual_topic_fields = set(rd["primary_topic"].keys()) if rd.get("primary_topic") else set()
    topic_ok = expected_topic_fields.issubset(actual_topic_fields)
    record("primary_topic_structure", topic_ok,
           f"expected={sorted(expected_topic_fields)}, actual={sorted(actual_topic_fields)}")
    
    # Check concepts structure
    expected_concept_fields = {"id", "wikidata", "display_name", "level", "score"}
    actual_concept_fields = set(rd["concepts"][0].keys()) if rd.get("concepts") else set()
    concept_ok = expected_concept_fields.issubset(actual_concept_fields)
    record("concepts_structure", concept_ok,
           f"expected={sorted(expected_concept_fields)}, actual={sorted(actual_concept_fields)}")
    
    # Check open_access structure
    expected_oa_fields = {"is_oa", "oa_status", "oa_url", "any_repository_has_fulltext"}
    actual_oa_fields = set(rd["open_access"].keys()) if rd.get("open_access") else set()
    oa_ok = expected_oa_fields.issubset(actual_oa_fields)
    record("open_access_structure", oa_ok,
           f"expected={sorted(expected_oa_fields)}, actual={sorted(actual_oa_fields)}")
    
    # Check concept IDs have the right prefix
    concept_ids = [c["id"] for c in rd.get("concepts", []) if c.get("id")]
    all_c_ids_ok = all(re.match(r"^https://openalex\.org/C\d+$", cid) for cid in concept_ids)
    record("concepts_id_prefix", all_c_ids_ok,
           f"Checked {len(concept_ids)} concept IDs in work")
    
    # Check referenced_works have the right prefix
    ref_works = rd.get("referenced_works", [])
    if ref_works:
        all_ref_ok = all(re.match(r"^https://openalex\.org/W\d+$", rw) for rw in ref_works[:20])
        record("referenced_works_prefix", all_ref_ok,
               f"Checked {min(len(ref_works), 20)} referenced_works IDs")
    else:
        record("referenced_works_prefix", True, "No referenced works (OK)")
else:
    record("nested_structure_checks", False, "Could not find a rich work for testing")

In [ ]:
print("=" * 60)
print("TEST 7: Null Checks on Required Fields")
print("=" * 60)

# Single aggregation for all null checks
REQUIRED_FIELDS = ["id", "type", "is_paratext"]
MOSTLY_REQUIRED = ["title", "publication_year", "updated_date", "created_date", "is_retracted"]

null_aggs = [F.count(F.when(F.col(f).isNull(), 1)).alias(f"null_{f}") for f in REQUIRED_FIELDS + MOSTLY_REQUIRED]
null_stats = works_df.agg(*null_aggs).first().asDict()

for field in REQUIRED_FIELDS:
    null_count = null_stats[f"null_{field}"]
    passed = null_count == 0
    record(f"not_null_{field}", passed,
           f"{null_count:,}/{works_total:,} null values")

for field in MOSTLY_REQUIRED:
    null_count = null_stats[f"null_{field}"]
    null_pct = null_count / works_total if works_total > 0 else 0
    passed = null_pct < 0.10
    record(f"mostly_not_null_{field}", passed,
           f"{null_count:,}/{works_total:,} ({null_pct:.2%}) null values")

In [ ]:
print("=" * 60)
print("TEST 8: Duplicate ID Check")
print("=" * 60)

DUP_CHECK_TABLES = [
    ("works",        "openalex.works.openalex_works_snapshot"),
    ("authors",      "openalex.authors.openalex_authors_snapshot"),
    ("institutions", "openalex.institutions.openalex_institutions_snapshot"),
    ("sources",      "openalex.sources.openalex_sources_snapshot"),
    ("publishers",   "openalex.publishers.openalex_publishers_snapshot"),
    ("funders",      "openalex.funders.openalex_funders_snapshot"),
    ("topics",       "openalex.common.openalex_topics_snapshot"),
    ("concepts",     "openalex.common.openalex_concepts_snapshot"),
    ("keywords",     "openalex.common.openalex_keywords_snapshot"),
]

for entity_name, table in DUP_CHECK_TABLES:
    df = works_df if table == "openalex.works.openalex_works_snapshot" else spark.read.table(table)
    total = works_total if table == "openalex.works.openalex_works_snapshot" else df.count()
    distinct = df.select("id").distinct().count()
    dups = total - distinct
    passed = dups == 0
    record(f"no_duplicates_{entity_name}", passed,
           f"{total:,} total, {distinct:,} distinct, {dups:,} duplicates")

## Test 9: S3 Manifest Validation
Verify that manifests were written to S3 for every entity and that their record counts
match the snapshot tables. This confirms the actual S3 export completed correctly.

In [ ]:
print("=" * 60)
print("TEST 9: S3 Manifest Validation")
print("=" * 60)

from datetime import datetime as _dt
date_str = _dt.now().strftime("%Y-%m-%d")
s3_base = f"s3://openalex-snapshots/full/{date_str}"

# All entities that should have manifests after a full snapshot export
MANIFEST_ENTITIES = [
    ("works",              "openalex.works.openalex_works_snapshot"),
    ("authors",            "openalex.authors.openalex_authors_snapshot"),
    ("institutions",       "openalex.institutions.openalex_institutions_snapshot"),
    ("sources",            "openalex.sources.openalex_sources_snapshot"),
    ("publishers",         "openalex.publishers.openalex_publishers_snapshot"),
    ("funders",            "openalex.funders.openalex_funders_snapshot"),
    ("awards",             "openalex.awards.openalex_awards_snapshot"),
    ("keywords",           "openalex.common.openalex_keywords_snapshot"),
    ("concepts",           "openalex.common.openalex_concepts_snapshot"),
    ("topics",             "openalex.common.openalex_topics_snapshot"),
    ("subfields",          "openalex.common.openalex_subfields_snapshot"),
    ("fields",             "openalex.common.openalex_fields_snapshot"),
    ("domains",            "openalex.common.openalex_domains_snapshot"),
    ("continents",         "openalex.common.openalex_continents_snapshot"),
    ("countries",          "openalex.common.openalex_countries_snapshot"),
    ("institution-types",  "openalex.common.openalex_institution_types_snapshot"),
    ("languages",          "openalex.common.openalex_languages_snapshot"),
    ("licenses",           "openalex.common.openalex_licenses_snapshot"),
    ("sdgs",               "openalex.common.openalex_sdgs_snapshot"),
    ("source-types",       "openalex.common.openalex_source_types_snapshot"),
    ("work-types",         "openalex.common.openalex_work_types_snapshot"),
]

print(f"  Checking manifests at: {s3_base}")

for entity_name, snapshot_table in MANIFEST_ENTITIES:
    manifest_path = f"{s3_base}/{entity_name}/manifest"

    try:
        manifest_raw = dbutils.fs.head(manifest_path, 10_000_000)
        manifest = json.loads(manifest_raw)
    except Exception as e:
        record(f"s3_manifest_{entity_name}", False, f"Manifest not found or unreadable: {e}")
        continue

    manifest_count = manifest.get("meta", {}).get("record_count", 0)
    manifest_size = manifest.get("meta", {}).get("content_length", 0)
    num_files = len(manifest.get("entries", []))

    if manifest_count == 0:
        record(f"s3_manifest_{entity_name}", False,
               f"Manifest exists but record_count=0 ({num_files} files, {manifest_size/(1024**2):.1f} MB)")
        continue

    # Use cached count for works, read table for others
    try:
        table_count = works_total if snapshot_table == "openalex.works.openalex_works_snapshot" else spark.read.table(snapshot_table).count()
        match = manifest_count == table_count
        detail = (f"manifest={manifest_count:,}  table={table_count:,}  "
                  f"files={num_files}  size={manifest_size/(1024**2):.1f} MB")
        if not match:
            detail += f"  MISMATCH (diff={abs(manifest_count - table_count):,})"
        record(f"s3_manifest_{entity_name}", match, detail)
    except Exception as e:
        record(f"s3_manifest_{entity_name}", True,
               f"manifest={manifest_count:,}  files={num_files}  size={manifest_size/(1024**2):.1f} MB (table check skipped: {e})")

## Test 10: S3 Partition Path Format
Verify that partition directories use clean date format (`updated_date=2026-02-24/`)
and not URL-encoded timestamps (`updated_date=2026-02-24 00%3A00%3A00/`).
Timestamp partitions are caused by the `_partition_date` column being TIMESTAMP instead of DATE.

In [ ]:
print("=" * 60)
print("TEST 10: S3 Partition Path Format")
print("=" * 60)

from datetime import datetime as _dt
date_str = _dt.now().strftime("%Y-%m-%d")
s3_base = f"s3://openalex-snapshots/full/{date_str}"

PARTITION_ENTITIES = [
    "works", "authors", "institutions", "sources", "publishers",
    "funders", "awards", "keywords", "concepts",
    "topics", "subfields", "fields", "domains",
    "continents", "countries", "institution-types",
    "languages", "licenses", "sdgs", "source-types", "work-types",
]

for entity_name in PARTITION_ENTITIES:
    entity_path = f"{s3_base}/{entity_name}"
    try:
        entries = dbutils.fs.ls(entity_path)
        partitions = [e.name for e in entries if e.name.startswith("updated_date=")]
        bad = [p for p in partitions if "%3A" in p or "00:00:00" in p]
        passed = len(bad) == 0
        detail = f"{len(partitions)} partitions"
        if bad:
            detail += f", {len(bad)} with timestamp format (e.g. {bad[0].rstrip('/')})"
        record(f"partition_format_{entity_name}", passed, detail)
    except Exception as e:
        record(f"partition_format_{entity_name}", False, f"Could not list: {e}")

## Test 11: Abstract Truncation Validity
Verify that truncated `abstract_inverted_index` values are still valid JSON.
Corrupt truncation (cut mid-key) would break downstream consumers.

In [ ]:
print("=" * 60)
print("TEST 11: Abstract Truncation Validity")
print("=" * 60)

from pyspark.sql.types import BooleanType

@udf(BooleanType())
def is_valid_json(s):
    if s is None:
        return True
    try:
        json.loads(s)
        return True
    except (json.JSONDecodeError, ValueError):
        return False

# Read from snapshot table (before abstract is parsed to MapType)
abstracts_df = spark.read.table("openalex.works.openalex_works_snapshot")

abstract_stats = abstracts_df.agg(
    F.count(F.when(F.col("abstract_inverted_index").isNotNull(), 1)).alias("total_abstracts"),
    F.count(F.when(
        F.col("abstract_inverted_index").isNotNull() & ~is_valid_json(F.col("abstract_inverted_index")),
        1
    )).alias("invalid_json"),
).first()

total_abs = abstract_stats["total_abstracts"]
invalid = abstract_stats["invalid_json"]
passed = invalid == 0
record("abstract_json_valid", passed,
       f"{invalid:,}/{total_abs:,} abstracts have invalid JSON")


## Test 12: Updated Date Recency
The most recent `updated_date` should be within the last 7 days.
If this fails, the pipeline may have stalled and exported stale data.

In [ ]:
print("=" * 60)
print("TEST 12: Updated Date Recency")
print("=" * 60)

from datetime import datetime, timedelta

max_updated = works_df.agg(F.max("updated_date")).first()[0]
days_ago = (datetime.now(max_updated.tzinfo) - max_updated).days if max_updated else None

passed = days_ago is not None and days_ago <= 7
record("updated_date_recency", passed,
       f"max updated_date={max_updated}, {days_ago} days ago [threshold: <= 7 days]")


## Test 13: Keywords & SDGs Population Rates
Check that keywords and sustainable development goals are populated at expected rates.

In [ ]:
print("=" * 60)
print("TEST 13: Keywords & SDGs Population Rates")
print("=" * 60)

pop_stats = works_df.agg(
    F.count("*").alias("total"),
    # Keywords
    F.count(F.when(F.size("keywords") > 0, 1)).alias("with_keywords"),
    # SDGs
    F.count(F.when(F.size("sustainable_development_goals") > 0, 1)).alias("with_sdgs"),
    # Non-xpac keywords (cleaner signal)
    F.count(F.when(F.col("is_xpac") == False, 1)).alias("non_xpac_total"),
    F.count(F.when((F.col("is_xpac") == False) & (F.size("keywords") > 0), 1)).alias("non_xpac_with_keywords"),
).first().asDict()

# Keywords: expect most non-xpac works to have keywords (they come from topics)
kw_pct = pop_stats["with_keywords"] / pop_stats["total"]
passed_kw = kw_pct >= 0.50
record("works_have_keywords", passed_kw,
       f'{pop_stats["with_keywords"]:,}/{pop_stats["total"]:,} ({kw_pct:.1%}) have keywords [threshold: 50%]')

nx_kw_pct = pop_stats["non_xpac_with_keywords"] / pop_stats["non_xpac_total"]
passed_nx_kw = nx_kw_pct >= 0.65
record("non_xpac_have_keywords", passed_nx_kw,
       f'{pop_stats["non_xpac_with_keywords"]:,}/{pop_stats["non_xpac_total"]:,} ({nx_kw_pct:.1%}) non-xpac have keywords [threshold: 65%]')

# SDGs: only a fraction of works have SDG tags
sdg_pct = pop_stats["with_sdgs"] / pop_stats["total"]
passed_sdg = sdg_pct >= 0.05
record("works_have_sdgs", passed_sdg,
       f'{pop_stats["with_sdgs"]:,}/{pop_stats["total"]:,} ({sdg_pct:.1%}) have SDGs [threshold: 5%]')


## Summary

In [ ]:
print("\n" + "=" * 60)
print("SMOKE TEST SUMMARY")
print("=" * 60)

passed_count = sum(1 for r in results if r["passed"])
failed_count = sum(1 for r in results if not r["passed"])
total_count = len(results)

print(f"\nTotal: {total_count} tests")
print(f"Passed: {passed_count}")
print(f"Failed: {failed_count}")

if failed_count > 0:
    print(f"\nFAILED TESTS:")
    for r in results:
        if not r["passed"]:
            print(f"  [XX] {r['test']}: {r['detail']}")
    print(f"\n{'!' * 60}")
    print(f"SNAPSHOT SMOKE TESTS FAILED - {failed_count} test(s) need attention")
    print(f"{'!' * 60}")
    # Raise to mark the Databricks task as failed
    raise Exception(f"Snapshot smoke tests failed: {failed_count}/{total_count} tests failed")
else:
    print(f"\nAll {total_count} tests passed. Snapshot looks good!")